# APD Kaggle production batch runner

Generates one deterministic APD production shard on Kaggle free GPU and leaves exactly one downloadable ZIP in `/kaggle/working`.

Scientific invariants are not changed: the notebook reads the committed manifest, uses the locked prompt grid and seeds, and does not modify APD, occupations, hypotheses, ground truth, or target sample size.

In [ ]:
# === Production batch configuration ===
import os

REPO_URL = "https://github.com/hlaverde/apd-audit.git"
REPO_BRANCH = "cloud-generation-20260602"
REPO_DIR = "/kaggle/working/apd-audit"

MODEL = "runwayml/stable-diffusion-v1-5"
LANGUAGE = "en"
SHARD_ID = 0
N_SHARDS = 4
MAX_IMAGES_PER_RUN = 300
RUN_LABEL = "sd15_en_s0"

GRID = "main"
RUNNER = "kaggle"
RUN_ID = RUN_LABEL
CHECKPOINT_EVERY = 10
CLASSIFY = True
DRY_RUN = False

OUTPUT_ROOT = f"/kaggle/working/apd_cloud_run_{RUN_LABEL}"
FINAL_ZIP = f"/kaggle/working/apd_cloud_run_{RUN_LABEL}.zip"

# Keep virtualenvs/caches out of /kaggle/working so Kaggle output is clean.
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/apd-venv"
os.environ["UV_LINK_MODE"] = "copy"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
os.environ["PIP_CACHE_DIR"] = "/tmp/pip-cache"
os.environ["HF_HOME"] = "/tmp/hf-cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf-cache"
os.environ["DIFFUSERS_CACHE"] = "/tmp/hf-cache"

print("RUN_LABEL:", RUN_LABEL)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("FINAL_ZIP:", FINAL_ZIP)


## Recommended next batches

| Label | Model | Language | Shard |
|---|---|---|---:|
| `sd15_en_s0` | `runwayml/stable-diffusion-v1-5` | `en` | 0/4 |
| `sd15_en_s1` | `runwayml/stable-diffusion-v1-5` | `en` | 1/4 |
| `sd15_en_s2` | `runwayml/stable-diffusion-v1-5` | `en` | 2/4 |
| `sd15_en_s3` | `runwayml/stable-diffusion-v1-5` | `en` | 3/4 |
| `sd15_esES_s0` | `runwayml/stable-diffusion-v1-5` | `es-ES` | 0/4 |
| `sd15_esLatAm_s0` | `runwayml/stable-diffusion-v1-5` | `es-LatAm` | 0/4 |
| `sd15_ptBR_s0` | `runwayml/stable-diffusion-v1-5` | `pt-BR` | 0/4 |
| `sdxl_en_s0` | `stabilityai/stable-diffusion-xl-base-1.0` | `en` | 0/4 |

For each batch, edit only `MODEL`, `LANGUAGE`, `SHARD_ID`, and `RUN_LABEL`. Keep `N_SHARDS=4` unless the manifest is regenerated with a different sharding plan.

In [ ]:
# Start from a clean Kaggle working directory except for any previous final ZIPs.
import pathlib, shutil

for target in [REPO_DIR, OUTPUT_ROOT]:
    p = pathlib.Path(target)
    if p.exists():
        shutil.rmtree(p)

zip_path = pathlib.Path(FINAL_ZIP)
if zip_path.exists():
    zip_path.unlink()

print("Clean workspace prepared")


In [ ]:
# Clone the exact branch that contains the cloud runner and manifest.
import os, pathlib, subprocess

subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
os.chdir(REPO_DIR)
print("Repo:", pathlib.Path.cwd())


In [ ]:
# Install uv and sync the project into /tmp/apd-venv, not /kaggle/working/.venv.
import os, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    [sys.executable, "-m", "uv", "sync", "--extra", "ml"],
    check=True,
    env=os.environ.copy(),
)

print("UV_PROJECT_ENVIRONMENT:", os.environ["UV_PROJECT_ENVIRONMENT"])


In [ ]:
# Preview the selected slice from the committed manifest.
import hashlib, pandas as pd, pathlib

manifest_path = pathlib.Path("results/missing_generation_manifest_2026-06-02.csv")
manifest = pd.read_csv(manifest_path)

def shard(image_id, n):
    return int(hashlib.sha256(str(image_id).encode("utf-8")).hexdigest()[:16], 16) % n

preview = manifest[
    (manifest.model == MODEL)
    & (manifest.language == LANGUAGE)
    & (manifest.grid == GRID)
    & (manifest.prompt_status == "ok")
].copy()
preview = preview[preview.image_id.map(lambda x: shard(x, N_SHARDS) == SHARD_ID)]

print("Rows in selected shard before local checkpoint filtering:", len(preview))
print("Rows requested this run:", min(MAX_IMAGES_PER_RUN, len(preview)))
display(preview.head(8))


In [ ]:
# Run production generation. Checkpoints are written into OUTPUT_ROOT.
import os, subprocess, sys

cmd = [
    sys.executable, "-m", "uv", "run", "python", "scripts/cloud_generation_runner.py",
    "--manifest", "results/missing_generation_manifest_2026-06-02.csv",
    "--existing-metadata", "images/main/metadata.parquet",
    "--output-root", OUTPUT_ROOT,
    "--runner", RUNNER,
    "--run-id", RUN_ID,
    "--model", MODEL,
    "--language", LANGUAGE,
    "--grid", GRID,
    "--shard-id", str(SHARD_ID),
    "--n-shards", str(N_SHARDS),
    "--max-images-per-run", str(MAX_IMAGES_PER_RUN),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
]
cmd.append("--classify" if CLASSIFY else "--no-classify")
if DRY_RUN:
    cmd.append("--dry-run")

print(" ".join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy())


In [ ]:
# Build the single final ZIP at /kaggle/working/apd_cloud_run_{RUN_LABEL}.zip.
# Remove any internal ZIP produced by the runner before packaging the final output.
import pathlib, shutil

output_root = pathlib.Path(OUTPUT_ROOT)
for internal_zip in output_root.rglob("*.zip"):
    internal_zip.unlink()

base = pathlib.Path(FINAL_ZIP).with_suffix("")
if pathlib.Path(FINAL_ZIP).exists():
    pathlib.Path(FINAL_ZIP).unlink()
archive = shutil.make_archive(str(base), "zip", root_dir=output_root)
print("Final ZIP:", archive)


In [ ]:
# Summarize generated artifact before cleanup.
import pathlib, pandas as pd, zipfile

zip_path = pathlib.Path(FINAL_ZIP)
metadata_files = sorted(pathlib.Path(OUTPUT_ROOT).glob("images/main/metadata_*.parquet"))
if not metadata_files:
    raise FileNotFoundError("No metadata shard parquet was produced")
metadata_path = metadata_files[-1]
metadata = pd.read_parquet(metadata_path)
image_count = int(len(metadata))
zip_size_mb = zip_path.stat().st_size / (1024 * 1024)

print("ZIP name:", zip_path.name)
print(f"ZIP size: {zip_size_mb:.2f} MiB")
print("Images generated:", image_count)
print("Metadata parquet included:", metadata_path.relative_to(OUTPUT_ROOT).as_posix())

with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    assert any(name.endswith(metadata_path.name) for name in names), metadata_path.name


In [ ]:
# Cleanup: leave exactly one downloadable ZIP under /kaggle/working.
import pathlib, shutil

for target in [REPO_DIR, OUTPUT_ROOT]:
    p = pathlib.Path(target)
    if p.exists():
        shutil.rmtree(p)

# Remove common cache folders if they accidentally appeared in /kaggle/working.
for junk in [".venv", ".cache", "uv-cache", "pip-cache", "hf-cache"]:
    p = pathlib.Path("/kaggle/working") / junk
    if p.exists():
        shutil.rmtree(p) if p.is_dir() else p.unlink()

remaining = sorted(p.name for p in pathlib.Path("/kaggle/working").iterdir())
expected = [pathlib.Path(FINAL_ZIP).name]
print("Remaining /kaggle/working entries:", remaining)
if remaining != expected:
    raise RuntimeError(f"Expected only {expected}, found {remaining}")


## Resume behavior

If Kaggle disconnects before the cleanup cell, rerun the notebook with the same configuration. The runner reads existing checkpoint metadata inside `OUTPUT_ROOT` and skips completed `image_id`s. If the final cleanup cell has already run, the batch is complete and the ZIP is ready to download.